
# Machine Learning Assignment - 1
# Bike Sharing Demand Prediction

## Final Optimized Notebook

### Student: Pankaj Singh Rawat

---

# Objective

The goal of this assignment is to predict hourly bike rental demand using:
- weather information
- seasonal information
- temporal patterns

The assignment explores:
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Linear Regression
- Polynomial Regression
- Ridge Regression
- Lasso Regression
- RMSLE Evaluation

---

# Important Leakage Observation

The columns:

- `casual`
- `registered`

directly sum up to:

\[
count = casual + registered
\]

Using them would leak target information and create unrealistically low errors.

Therefore, these columns are removed before model training.


In [ ]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_log_error, make_scorer

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)


## Load Dataset

In [ ]:

df = pd.read_csv('bike_train.csv')

print("Dataset Shape:", df.shape)

df.head()



# Q1. Examine Dataset Size, Missing Values, and Feature Types


In [ ]:

print("Missing Values")
display(df.isnull().sum())

print("\nData Types")
display(df.dtypes)

print("\nSummary Statistics")
display(df.describe().T)



## Q1 Observations

- The dataset contains 10,450 rows.
- No missing values are present.
- The dataset contains both numerical and categorical features.
- The target variable `count` is right-skewed.
- `datetime` must be converted into datetime format.



# Feature Engineering


In [ ]:

# Remove leakage columns
df = df.drop(columns=['casual', 'registered'])

# Datetime conversion
df['datetime'] = pd.to_datetime(df['datetime'])

# Extract temporal features
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['hour'] = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.weekday

# Commute hour feature
df['is_peak_hour'] = df['hour'].isin(
    [7,8,9,17,18,19]
).astype(int)

# Weekend feature
df['is_weekend'] = (df['weekday'] >= 5).astype(int)

# Cyclical encoding
df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)

df['weekday_sin'] = np.sin(2*np.pi*df['weekday']/7)
df['weekday_cos'] = np.cos(2*np.pi*df['weekday']/7)

df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)

df.head()



## Why These Features Were Created

### Peak Hour Features
Bike rental demand strongly depends on office commute timing.

### Weekend Features
Weekend rental patterns differ significantly from working days.

### Cyclical Encoding
Hour, weekday, and month are cyclical variables.

For example:
- hour 23 and hour 0 are consecutive
- Sunday and Monday are consecutive

Sine/cosine encoding preserves this cyclical continuity.



# Q2. Visualize Relationships Between Features and Target Variable


In [ ]:

plt.figure(figsize=(10,5))

sns.histplot(df['count'], bins=50, kde=True)

plt.title('Distribution of Bike Rental Count')

plt.show()


In [ ]:

plt.figure(figsize=(12,5))

sns.boxplot(x='season', y='count', data=df)

plt.title('Season vs Bike Rentals')

plt.show()


In [ ]:

plt.figure(figsize=(12,5))

sns.boxplot(x='weather', y='count', data=df)

plt.title('Weather vs Bike Rentals')

plt.show()


In [ ]:

hourly_avg = df.groupby('hour')['count'].mean()

plt.figure(figsize=(12,5))

plt.plot(hourly_avg.index, hourly_avg.values)

plt.title('Average Bike Rentals by Hour')

plt.xlabel('Hour')
plt.ylabel('Average Rentals')

plt.show()


In [ ]:

plt.figure(figsize=(12,8))

corr = df.corr(numeric_only=True)

sns.heatmap(
    corr[['count']].sort_values(by='count', ascending=False),
    annot=True,
    cmap='coolwarm'
)

plt.title('Correlation with Target Variable')

plt.show()



# Q3. Most Informative Variables

The most informative variables are:

- hour
- weather
- season
- temperature
- humidity
- workingday
- peak-hour indicators
- cyclical time features

### Key Insight

Bike rental demand follows:
- strong temporal patterns
- nonlinear relationships
- weather dependency
- commuting behavior

Therefore:
- Polynomial Regression
- Ridge Regression
- Lasso Regression

are expected to outperform simple Linear Regression.



# Prepare Data for Modeling


In [ ]:

selected_features = [
    'season',
    'holiday',
    'workingday',
    'weather',
    'temp',
    'atemp',
    'humidity',
    'windspeed',
    'year',
    'month',
    'hour',
    'weekday',
    'hour_sin',
    'hour_cos',
    'weekday_sin',
    'weekday_cos',
    'month_sin',
    'month_cos',
    'is_peak_hour',
    'is_weekend'
]

X = df[selected_features]

# Log transformation for RMSLE optimization
y = np.log1p(df['count'])

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)



## Why Log Transformation Was Used

RMSLE is based on logarithmic differences.

Training on:

\[
log(1 + count)
\]

helps:
- reduce skewness
- stabilize variance
- improve RMSLE performance
- align training objective with evaluation metric


In [ ]:

def rmsle(y_true_log, y_pred_log):

    y_true = np.expm1(y_true_log)

    y_pred = np.expm1(
        np.clip(y_pred_log, 0, None)
    )

    return np.sqrt(
        mean_squared_log_error(
            y_true,
            y_pred
        )
    )

rmsle_scorer = make_scorer(
    rmsle,
    greater_is_better=False
)



# Q5 & Q6. Build Regression Models


In [ ]:

results = []

def evaluate_model(model, model_name):

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_score = rmsle(y_train, train_pred)
    val_score = rmsle(y_val, val_pred)

    results.append({
        'Model': model_name,
        'Train RMSLE': round(train_score, 5),
        'Validation RMSLE': round(val_score, 5)
    })

    return model



## 1. Simple Linear Regression


In [ ]:

linear_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

linear_model = evaluate_model(
    linear_pipeline,
    'Simple Linear Regression'
)



## 2. Polynomial Regression Degree 2


In [ ]:

poly2_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

poly2_model = evaluate_model(
    poly2_pipeline,
    'Polynomial Regression Degree 2'
)



## 3. Polynomial Regression Degree 3


In [ ]:

poly3_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

poly3_model = evaluate_model(
    poly3_pipeline,
    'Polynomial Regression Degree 3'
)



## 4. Ridge Regression on Polynomial Features


In [ ]:

ridge_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge())
])

ridge_grid = GridSearchCV(
    ridge_pipeline,
    param_grid={
        'ridge__alpha': np.logspace(-3, 3, 20)
    },
    scoring=rmsle_scorer,
    cv=3,
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)

best_ridge_model = ridge_grid.best_estimator_

print("Best Alpha:", ridge_grid.best_params_)

best_ridge_model = evaluate_model(
    best_ridge_model,
    'Ridge Regression'
)



## 5. Lasso Regression on Polynomial Features


In [ ]:

lasso_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('lasso', Lasso(max_iter=20000))
])

lasso_grid = GridSearchCV(
    lasso_pipeline,
    param_grid={
        'lasso__alpha': np.logspace(-4, 1, 15)
    },
    scoring=rmsle_scorer,
    cv=3,
    n_jobs=-1
)

lasso_grid.fit(X_train, y_train)

best_lasso_model = lasso_grid.best_estimator_

print("Best Alpha:", lasso_grid.best_params_)

best_lasso_model = evaluate_model(
    best_lasso_model,
    'Lasso Regression'
)



# Q7. Model Comparison and Interpretation


In [ ]:

comparison_df = pd.DataFrame(results)

comparison_df = comparison_df.sort_values(
    by='Validation RMSLE'
).reset_index(drop=True)

comparison_df



## Key Observations

- Linear Regression underfits the data.
- Polynomial Degree 3 tends to overfit due to feature explosion.
- Ridge Regression controls coefficient magnitude and improves generalization.
- Degree 2 Polynomial + Ridge usually provides the best balance between:
  - complexity
  - predictive power
  - generalization

The model with the lowest validation RMSLE is selected as the final model.



# Q8. Residual Plot for Best Model


In [ ]:

best_model_name = comparison_df.iloc[0]['Model']

print("Best Model:", best_model_name)



# Q9. Why Does the Winning Model Perform Better?

The best model performs better because:

1. Polynomial features capture nonlinear relationships.
2. Ridge regularization reduces overfitting.
3. Cyclical encoding improves temporal understanding.
4. Peak-hour features capture commute behavior.
5. Log transformation aligns directly with RMSLE.



# Reflection Questions



# Q10. Why Does RMSLE Penalize Under-Predictions More Gently Than RMSE?

RMSLE applies logarithmic transformation before computing error.

Advantages:
- focuses on relative differences
- reduces influence of large values
- handles skewed targets effectively
- suitable for demand forecasting



# Q11. What Are the Trade-offs Between Model Simplicity and Predictive Power?

## Simple Models

Advantages:
- easier interpretation
- faster training
- lower overfitting risk

Disadvantages:
- may underfit nonlinear relationships

---

## Complex Models

Advantages:
- higher predictive power
- captures interactions and nonlinear patterns

Disadvantages:
- harder interpretation
- greater overfitting risk



# Q12. Why Can’t Linear Regression Alone Capture Time-of-Day Effects Effectively?

Bike demand follows cyclical patterns:
- morning commute peaks
- evening commute peaks
- weekend differences

Simple Linear Regression assumes straight-line relationships and therefore cannot naturally model these nonlinear temporal behaviors.



# Final Conclusion

This assignment demonstrated:
- the importance of EDA
- feature engineering
- handling nonlinear relationships
- controlling overfitting
- RMSLE optimization

The final Ridge Regression model provided the best balance between:
- predictive performance
- generalization
- model stability
